# Preferential-flow sensitivity analysis

This notebook reproduces:

- **Figure 4.15**, comparing surface-soil properties between samples with and without preferential flow.
- **Table A.4**, reporting two-sided Mann–Whitney U tests.

The analysis is restricted to **Layer 1**. The variables are \(K_{fs}\), macrofauna biomass, SOC, ESP, laboratory-measured EC\(_{1:1}\), and MIR-predicted EC\(_{1:2}\).

## Important plotting detail

The highest laboratory EC\(_{1:1}\) value in the preferential-flow group is omitted **only from panel (e) of the published figure**. It remains included in the Mann–Whitney test and in Table A.4. No observations are removed from the statistical analysis.


In [ ]:
# ============================================================
# PREFERENTIAL FLOW — CLEAN FIGURE + MANN–WHITNEY SUMMARY
# Layer 1 only
#
# Updates:
# - Original Excel column "Ksat" is displayed as Kfs in labels/tables/figure.
# - Panel e) removes the highest EC1:1 lab outlier in "Yes"
#   only for plotting, not for Mann–Whitney tests.
# - Panel f) EC1:2 spectral remains unchanged.
# - Bigger letters, numbers, points and legend.
# ============================================================

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

# ------------------------------------------------------------
# 1. PROJECT PATHS AND DATA
# ------------------------------------------------------------
from pathlib import Path

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

file_path = PROJECT_DIR / "data" / "soil_data.xlsx"
sheet_name = "PredictionNewSamplesSalinity_Fi"

out_dir = PROJECT_DIR / "outputs" / "09_preferential_flow"
out_dir.mkdir(parents=True, exist_ok=True)

if not file_path.exists():
    raise FileNotFoundError(
        f"Input file not found: {file_path}\n"
        "Place the soil dataset in data/soil_data.xlsx or update file_path."
    )

df = pd.read_excel(file_path, sheet_name=sheet_name)
df.columns = df.columns.str.strip()

# ------------------------------------------------------------
# 3. COLUMN NAMES
# ------------------------------------------------------------
possible_preferential_cols = [
    "Preferential flow",
    "Prefrential flow",
    "Preferential Flow",
    "preferential flow",
    "Preferential_flow",
    "PreferentialFlow"
]

preferential_col = None

for col in possible_preferential_cols:
    if col in df.columns:
        preferential_col = col
        break

if preferential_col is None:
    raise ValueError(
        "Preferential flow column not found. Check the exact column name in the Excel file."
    )

print("Using preferential flow column:", preferential_col)

rename_dict = {
    "EC 1:2 (dS/cm)": "EC_spectral",
    "EC (dS/m) 1:1": "EC_lab",
    "ESP est": "ESP",
    "SOC (%)": "SOC",
    "Clay (%)": "Clay",
    "CEC cmolc/kg": "CEC",
    "TotalMacrofaunaBiomass (g)": "MacrofaunaBiomass_g"
}

df = df.rename(columns=rename_dict)

df["Zone"] = df["SITE"].astype(str).str.strip().str.upper()
df = df[df["Zone"].isin(["PERKERRA", "TANA"])].copy()

# NOTE:
# The original Excel column is still called "Ksat".
# Internally we keep "Ksat" for compatibility, but display it as Kfs.
numeric_cols = [
    "Layer",
    "Ksat",
    "MacrofaunaBiomass_g",
    "SOC",
    "ESP",
    "EC_lab",
    "EC_spectral",
    "pH",
    "Clay",
    "CEC"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# ------------------------------------------------------------
# 4. LAYER 1 ONLY
# ------------------------------------------------------------
df_l1 = df[df["Layer"] == 1].copy()

df_l1["PreferentialFlow_raw"] = (
    df_l1[preferential_col]
    .astype(str)
    .str.strip()
    .str.lower()
)

yes_values = ["yes", "y", "si", "sí", "1", "true", "present"]
no_values = ["no", "n", "0", "false", "absent"]

df_l1["PreferentialFlow"] = np.nan
df_l1.loc[df_l1["PreferentialFlow_raw"].isin(yes_values), "PreferentialFlow"] = 1
df_l1.loc[df_l1["PreferentialFlow_raw"].isin(no_values), "PreferentialFlow"] = 0

df_l1 = df_l1[df_l1["PreferentialFlow"].notna()].copy()
df_l1["PreferentialFlow"] = df_l1["PreferentialFlow"].astype(int)

df_l1["Preferential flow"] = df_l1["PreferentialFlow"].map({
    0: "No",
    1: "Yes"
})

print("\nPreferential flow counts:")
print(pd.crosstab(df_l1["Zone"], df_l1["Preferential flow"]))

# ------------------------------------------------------------
# 5. VARIABLES FOR FINAL FIGURE AND TESTS
# ------------------------------------------------------------
plot_vars = [
    "Ksat",
    "MacrofaunaBiomass_g",
    "SOC",
    "ESP",
    "EC_lab",
    "EC_spectral"
]

plot_vars = [v for v in plot_vars if v in df_l1.columns]

# Labels for figure axes
figure_labels = {
    "Ksat": r"$K_{\mathrm{fs}}$ (cm min$^{-1}$)",
    "MacrofaunaBiomass_g": "Macrofauna biomass (g)",
    "SOC": "SOC (%)",
    "ESP": "ESP (%)",
    "EC_lab": r"EC$_{1:1}$ (dS m$^{-1}$)",
    "EC_spectral": r"MIR-predicted EC$_{1:2}$ (dS m$^{-1}$)"
}

# Labels for exported Mann–Whitney table
table_labels = {
    "Ksat": "Kfs (cm min^-1)",
    "MacrofaunaBiomass_g": "Macrofauna biomass (g)",
    "SOC": "SOC (%)",
    "ESP": "ESP (%)",
    "EC_lab": "EC1:1 lab (dS m^-1)",
    "EC_spectral": "MIR-predicted EC1:2 (dS m^-1)"
}

# ------------------------------------------------------------
# 6. MANN–WHITNEY SUMMARY TABLE
# ------------------------------------------------------------
# IMPORTANT:
# Mann–Whitney tests are calculated using the full data.
# The EC1:1 lab outlier is removed only from panel e) in the figure, not here.

mw_rows = []

for var in plot_vars:
    temp = df_l1[[var, "PreferentialFlow"]].dropna()

    no = temp[temp["PreferentialFlow"] == 0][var]
    yes = temp[temp["PreferentialFlow"] == 1][var]

    if len(no) >= 3 and len(yes) >= 3:
        stat, p = mannwhitneyu(yes, no, alternative="two-sided")

        mw_rows.append({
            "Variable": table_labels.get(var, var),
            "N_No": len(no),
            "N_Yes": len(yes),
            "Median_No": no.median(),
            "Median_Yes": yes.median(),
            "Mean_No": no.mean(),
            "Mean_Yes": yes.mean(),
            "Difference_median_Yes_minus_No": yes.median() - no.median(),
            "U_statistic": stat,
            "p_value": p
        })

mw_summary = pd.DataFrame(mw_rows)

round_cols = [
    "Median_No",
    "Median_Yes",
    "Mean_No",
    "Mean_Yes",
    "Difference_median_Yes_minus_No",
    "U_statistic",
    "p_value"
]

for col in round_cols:
    if col in mw_summary.columns:
        mw_summary[col] = mw_summary[col].round(4)

mw_summary.to_excel(
    out_dir / "table_A_4_mann_whitney_preferential_flow.xlsx",
    index=False
)

print("\nMann–Whitney summary:")
try:
    display(mw_summary)
except NameError:
    print(mw_summary)

# ------------------------------------------------------------
# 7. IDENTIFY PANEL e) OUTLIER TO REMOVE ONLY FROM FIGURE
# ------------------------------------------------------------
# Removes the highest EC1:1 lab value among "Yes" preferential-flow samples.
# This affects panel e) only and does not affect Mann–Whitney statistics.
# Panel f), EC1:2 spectral, remains unchanged.

remove_ec11_yes_outlier_from_plot = True
ec11_yes_outlier_index = None
ec11_yes_outlier_value = None

if remove_ec11_yes_outlier_from_plot and "EC_lab" in df_l1.columns:
    yes_ec11 = df_l1[
        (df_l1["PreferentialFlow"] == 1) &
        (df_l1["EC_lab"].notna())
    ]

    if len(yes_ec11) > 0:
        ec11_yes_outlier_index = yes_ec11["EC_lab"].idxmax()
        ec11_yes_outlier_value = yes_ec11.loc[ec11_yes_outlier_index, "EC_lab"]

        print(
            "\nPanel e) plotting only: removing highest EC1:1 lab value "
            f"from Yes group. Index = {ec11_yes_outlier_index}, "
            f"value = {ec11_yes_outlier_value:.4f}"
        )

# ------------------------------------------------------------
# 8. CLEAN MULTIPANEL FIGURE
# ------------------------------------------------------------
sns.set_style("whitegrid")
sns.set_context("talk", font_scale=1.15)

# Global font sizes
FONT_AXIS_LABEL = 22
FONT_TICK = 19
FONT_PANEL = 25
FONT_LEGEND = 20
FONT_LEGEND_TITLE = 21

plt.rcParams.update({
    "font.size": 20,
    "axes.labelsize": FONT_AXIS_LABEL,
    "xtick.labelsize": FONT_TICK,
    "ytick.labelsize": FONT_TICK,
    "legend.fontsize": FONT_LEGEND,
    "legend.title_fontsize": FONT_LEGEND_TITLE,
    "axes.linewidth": 1.1,
})

palette_points = {
    "PERKERRA": "#4C72B0",
    "TANA": "#DD8452"
}

fig, axes = plt.subplots(2, 3, figsize=(19.2, 11.8))
axes = axes.flatten()

letters = ["a)", "b)", "c)", "d)", "e)", "f)"]

for i, var in enumerate(plot_vars):
    ax = axes[i]

    # Use full data for all panels except panel e), EC1:1 lab.
    plot_data = df_l1.copy()

    if (
        var == "EC_lab" and
        ec11_yes_outlier_index is not None
    ):
        plot_data = plot_data.drop(index=ec11_yes_outlier_index)

    sns.boxplot(
        data=plot_data,
        x="Preferential flow",
        y=var,
        order=["No", "Yes"],
        color="#B8C7DC",
        width=0.58,
        fliersize=0,
        linewidth=1.5,
        ax=ax
    )

    sns.stripplot(
        data=plot_data,
        x="Preferential flow",
        y=var,
        order=["No", "Yes"],
        hue="Zone",
        palette=palette_points,
        dodge=False,
        alpha=0.78,
        size=7.2,
        edgecolor="white",
        linewidth=0.45,
        ax=ax
    )

    # Keep original values but display skewed variables on log-scale
    if var in ["Ksat", "MacrofaunaBiomass_g"]:
        ax.set_yscale("log")

    ax.set_xlabel("Preferential flow", fontsize=FONT_AXIS_LABEL)
    ax.set_ylabel(figure_labels.get(var, var), fontsize=FONT_AXIS_LABEL)

    ax.tick_params(
        axis="both",
        labelsize=FONT_TICK,
        width=1.1,
        length=5
    )

    ax.text(
        0.02,
        0.98,
        letters[i],
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=FONT_PANEL,
        fontweight="bold"
    )

    if ax.get_legend() is not None:
        ax.get_legend().remove()

# Remove unused axes if any
for j in range(len(plot_vars), len(axes)):
    fig.delaxes(axes[j])

# Shared legend
handles, legend_labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles[:2],
    legend_labels[:2],
    title="Zone",
    loc="upper center",
    ncol=2,
    frameon=False,
    fontsize=FONT_LEGEND,
    title_fontsize=FONT_LEGEND_TITLE,
    bbox_to_anchor=(0.5, 1.035),
    handletextpad=0.6,
    columnspacing=1.4,
    markerscale=1.5
)

# More spacing to avoid overlap
plt.tight_layout(rect=[0, 0, 1, 0.93])

# Additional spacing between rows and columns
fig.subplots_adjust(
    wspace=0.32,
    hspace=0.33
)

# ------------------------------------------------------------
# 9. SAVE FIGURE
# ------------------------------------------------------------
png_path = out_dir / "figure_4_15_preferential_flow.png"
tiff_path = out_dir / "figure_4_15_preferential_flow.tiff"
pdf_path = out_dir / "figure_4_15_preferential_flow.pdf"

plt.savefig(png_path, dpi=600, bbox_inches="tight")
plt.savefig(tiff_path, dpi=600, bbox_inches="tight")
plt.savefig(pdf_path, bbox_inches="tight")

plt.show()

print("\nDone. Outputs saved in:")
print(out_dir)

print("\nFigure files:")
print(png_path)
print(tiff_path)
print(pdf_path)